In [1]:

import sys, os
sys.path.append(os.path.abspath("..")) 
import json
import torch
from datasets import CNFDataset
from models import LightningModelCNF
from utils import args_cnf, gen_image_cnf, plotly_generate

**Read configuration files and arguments:**

In [4]:
# Arguments
parser = args_cnf()
args, unknown = parser.parse_known_args()

args.particle = "proton_exiting"
args.metadata_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/metadata.pkl"
args.dataset_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/{}/{}/{}/{}.zip"
args.cnf_ind_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/gan_ind.pkl"
args.save_dir = "/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/"
args.checkpoint_path = "/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/test_spline_noexiting/checkpoints"
args.checkpoint_name = args.particle

if args.particle == "muon" or args.particle == "proton_exiting":
    args.label_size = 7
elif args.particle == "proton_contained":
    args.label_size = 7
args.epochs = 50
args.log_every_n_steps = 2000
args.batch_size = 512
args.hidden = 256
args.num_workers = 64

**Load the pre-trained weights of the different generative-adversarial-network (GAN) models:**

In [5]:
# Dataset and generator models
test_set_p = CNFDataset(args, split="val")

checkpoint_path = "/".join((args.checkpoint_path, args.checkpoint_name, "train_loss", "last.ckpt"))


model = LightningModelCNF.load_from_checkpoint(checkpoint_path, img_shape = (args.img_size, args.img_size, args.img_size), 
                                    label_size = args.label_size, 
                                    hidden_features = args.hidden, 
                                    num_blocks_in_MADE = args.num_blocks_in_MADE, 
                                    num_transformers = args.num_transformers, 
                                    lr = args.lr, 
                                    wd = args.weight_decay)

model.eval()

# move to cpu
model.to("cpu")


LightningModelCNF(
  (nflow): Flow(
    (_transform): CompositeTransform(
      (_transforms): ModuleList(
        (0): MaskedPiecewiseRationalQuadraticAutoregressiveTransform(
          (autoregressive_net): MADE(
            (initial_layer): MaskedLinear(in_features=125, out_features=256, bias=True)
            (context_layer): Linear(in_features=7, out_features=256, bias=True)
            (activation): ReLU()
            (blocks): ModuleList(
              (0-1): 2 x MaskedResidualBlock(
                (context_layer): Linear(in_features=7, out_features=256, bias=True)
                (linear_layers): ModuleList(
                  (0-1): 2 x MaskedLinear(in_features=256, out_features=256, bias=True)
                )
                (activation): ReLU()
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (final_layer): MaskedLinear(in_features=256, out_features=3625, bias=True)
          )
        )
        (1): BatchNorm()
        (2)

In [ ]:
print(checkpoint_p['state_dict'].keys())

NameError: name 'checkpoint_p' is not defined

In [ ]:
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['max'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['min'])

2658
3


In [ ]:
for key, value in checkpoint_p['state_dict'].items():
    print(key, value.shape)

generator.bert.cls_token torch.Size([1, 1, 64])
generator.bert.embedding.input.weight torch.Size([64, 1])
generator.bert.embedding.input.bias torch.Size([64])
generator.bert.embedding.label.embedding.weight torch.Size([64, 10])
generator.bert.embedding.label.embedding.bias torch.Size([64])
generator.bert.embedding.position.vol_idx torch.Size([126])
generator.bert.embedding.position.embedding.weight torch.Size([126, 64])
generator.bert.embedding.noise.weight torch.Size([64, 512])
generator.bert.embedding.noise.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.0.weight torch.Size([64, 64])
generator.bert.transformer_blocks.0.attention.linear_layers.0.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.1.weight torch.Size([64, 64])
generator.bert.transformer_blocks.0.attention.linear_layers.1.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.2.weight torch.Size([64, 64])
generator.bert.transforme

In [ ]:
print(checkpoint_p['state_dict']['critic.bert.embedding.input.weight'])

tensor([[ 0.5004],
        [-2.5577],
        [ 0.5532],
        [-9.0691],
        [ 2.0405],
        [ 1.9576],
        [ 0.0836],
        [ 0.5649],
        [-0.5833],
        [-0.5987],
        [ 1.0329],
        [ 5.5730],
        [ 1.9099],
        [ 4.0439],
        [ 0.7667],
        [ 1.3041],
        [ 0.2591],
        [ 1.5450],
        [ 6.0901],
        [ 2.3652],
        [ 3.0429],
        [-3.6318],
        [ 2.1024],
        [-0.3306],
        [ 0.4442],
        [ 5.3540],
        [ 5.7043],
        [-0.4855],
        [ 2.4442],
        [-0.2827],
        [-0.6130],
        [ 0.6491],
        [-1.9083],
        [ 1.0670],
        [-0.6086],
        [ 0.6371],
        [ 1.2413],
        [-1.5205],
        [ 5.3508],
        [ 0.5478],
        [-3.5981],
        [-1.0268],
        [ 0.4822],
        [ 0.7821],
        [-2.2264],
        [ 1.6432],
        [ 1.1424],
        [ 1.7548],
        [ 0.8087],
        [ 2.2181],
        [ 0.6237],
        [ 0.8127],
        [ 1.

**Run each GAN on some arbitrary input kinematics:**

In [6]:
'''
Proton GAN
'''
import numpy as np

# Set your kinematics here:
# ke = 30.3  # Initial kinetic energy
# ini_dir = [0.9999999999999999, 0.0, 0.0]  # Initial direction
# ini_pos = [-1.5, -4.2, 2.7]  # Initial 3D position (mm)

# get one event from the test set
event = test_set_p[300]
# convert torch tensor to numpy array
ke = float(event['ke'].numpy())
ini_pos = event['pos_ini'].numpy()
ini_dir = event['dir_ini'].numpy()
if args.particle == "proton_exiting" or args.particle == "muon":
    exit_pos = event['pos_exit'].numpy()
else:
    exit_pos = None
img = event['image'].numpy()

#print(ke, ini_pos, ini_dir, exit_pos, img)

if False:
#if exit_pos is not None:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2], exit_pos[0], exit_pos[1], exit_pos[2]])
else:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2]])

#params = np.array([0.1925,  0.2249, -0.0153, -0.1201, -0.4700, -0.4900, -0.7400])
labels = torch.tensor([params], dtype=torch.float32)

print(labels)




tensor([[ 0.1700, -0.0323, -0.0461,  0.1032,  0.7060, -0.1773,  0.6857]])


/tmp/ipykernel_946004/1903654378.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ke = float(event['ke'].numpy())
/tmp/ipykernel_946004/1903654378.py:32: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1758491028874/work/torch/csrc/utils/tensor_new.cpp:253.)
  labels = torch.tensor([params], dtype=torch.float32)


In [7]:
with torch.no_grad():
    generated_p = model.sample(labels)

print(generated_p)

tensor([[[-0.9998, -0.9998, -0.9998, -0.9996, -0.9998, -0.9997, -0.9999,
          -1.0000, -0.9995, -0.9994, -0.9995, -0.9998, -0.9995, -0.9996,
          -0.9997, -0.9995, -0.9997, -0.9997, -0.9996, -0.9997, -0.9998,
          -0.9999, -0.9998, -0.9995, -0.9997, -0.9997, -1.0000, -0.9996,
          -1.0000, -0.9999, -0.9995, -0.9995, -0.9996, -0.9998, -0.9999,
          -0.9997, -0.9997, -0.9998, -0.9999, -0.9998, -0.9996, -0.9995,
          -0.9998, -0.9997, -0.9999, -0.9999, -1.0000, -0.9995, -0.9997,
          -0.9995, -0.9996, -0.9997, -0.9999, -0.9995, -0.9999, -0.9998,
          -0.9997, -0.9998, -0.9997, -1.0000, -0.9994, -0.9995, -0.9581,
          -0.9999, -0.9996, -0.9998, -0.9997, -0.9995, -0.9995, -0.9996,
          -0.9995, -1.0000, -0.9994, -0.9998, -0.9995, -0.9998, -0.9996,
          -0.9996, -0.9999, -0.9995, -0.9999, -0.9995, -0.9999, -0.9889,
          -0.9996, -1.0000, -0.9996, -0.9328, -0.8306, -0.9996, -0.9999,
          -0.9999, -0.9998, -0.9998, -0.9994, -0.99

tensor([[[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,  39.9746,   3.0000],
         [  3.0000,   3.0000, 230.2248,   3.0000,   3.0000],
         [  3.0000,   3.0000, 100.7135,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000, 228.2390,   3.0000,   3.0000],
         [  3.0000,   3.0000, 187.8737,   3.3160,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000

**Visualise the GAN-generated images:**

In [8]:
'''
Plot the generated images!
'''


#generated_p = generated_p.numpy()
# copy the image to a pure numpy array

print(generated_p.shape)
generated_p_1 = generated_p[0][0]

min_charge = 0
max_charge = test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']

generated_p_1 = (generated_p_1 + 1) / 2
generated_p_1 *= (max_charge - min_charge)
generated_p_1 += min_charge

print(generated_p_1.shape)
# reshape the generated image to a 5x5x5 array
generated_p_1 = generated_p_1.reshape(5, 5, 5)



generated_plot = np.zeros((5, 5, 5))

for i in range(generated_plot.shape[0]):
    for j in range(generated_plot.shape[1]):
        for k in range(generated_plot.shape[2]):
            generated_plot[i, j, k] = float(generated_p_1[i, j, k])
print(generated_plot)
# check the type of the elements in the array
print(generated_plot.dtype)

# Max deposited energy in one voxel
max_energy = generated_plot.max()

#generated_plot[generated_plot > 150] = 0
generated_plot[generated_plot < 0] = 0
print(max_energy)




torch.Size([1, 1, 125])
torch.Size([125])
[[[3.49593043e-01 3.80005360e-01 3.07491302e-01 6.85713530e-01
   3.55734944e-01]
  [5.56634545e-01 2.04069614e-01 2.80348063e-02 7.57633209e-01
   9.52192783e-01]
  [8.26680064e-01 3.09472561e-01 8.41737628e-01 5.89523435e-01
   4.42514062e-01]
  [7.82696128e-01 4.86101747e-01 5.00267744e-01 6.09137893e-01
   4.95314598e-01]
  [3.29285145e-01 1.70388222e-01 3.92487288e-01 8.85721564e-01
   4.92838025e-01]]

 [[4.44594383e-01 6.43908978e-02 7.13550210e-01 2.23882198e-02
   2.00701475e-01]
  [7.56939769e-01 9.03453827e-01 6.73429728e-01 3.76340032e-01
   1.62859440e-01]
  [4.51429725e-01 4.67477918e-01 2.77970552e-01 1.33635879e-01
   3.59598398e-01]
  [7.21574306e-01 7.92998672e-01 3.11552882e-01 4.62623835e-01
   1.46712184e-01]
  [1.71279788e-01 6.83534145e-03 7.58425713e-01 4.49151278e-01
   7.88441777e-01]]

 [[6.84821963e-01 5.16018748e-01 1.11346722e-01 7.93394923e-01
   1.08672023e-01]
  [3.16704154e-01 4.98880863e-01 3.98133874e-01 5.67

In [ ]:
generated_plot = generated_plot.astype(np.float32)

In [9]:

print("- Proton image:")
plotly_generate(generated_plot, max_energy=max_energy)


- Proton image:


In [10]:
event = test_set_p[300]
true_img = np.zeros((125,))

for i in range(true_img.shape[0]):
    true_img[i] = float(event["image"][i])

true_img = true_img.reshape(5, 5, 5)

# get back the normalization
min_charge = 0
max_charge = test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']
true_img = (true_img + 1) / 2
true_img *= (max_charge - min_charge)
true_img += min_charge

print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['std'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['mean'])

max_energy = true_img.max()

print(true_img)

print("- True image:")
plotly_generate(true_img, max_energy=max_energy)



303.3067761656923
197.1132585084357
[[[4.89625085e-01 7.85505549e-01 7.48636469e-01 8.99612826e-01
   2.67130833e-01]
  [6.70943745e-01 8.62628909e-01 6.15538061e-03 7.11219904e-02
   3.50310419e-01]
  [8.70494677e-01 2.31886221e-01 6.43868867e-01 1.11088914e-01
   4.53548050e-01]
  [6.94551964e-01 8.01972829e-01 5.95019333e-01 9.32323368e-01
   3.78090212e-01]
  [4.17851658e-01 3.70929688e-01 8.58924672e-01 2.14518374e-01
   8.99685853e-01]]

 [[4.94311104e-01 1.65428440e-01 6.48726509e-01 5.94411140e-01
   1.43330792e-01]
  [7.64599759e-01 3.73854152e-01 3.30745008e-01 7.38517611e-01
   7.33424004e-01]
  [3.42555807e-01 1.37090316e-01 4.83615793e-01 8.55477783e-01
   4.33209558e-01]
  [5.72119032e-01 4.06271863e-02 7.10074060e-02 3.81290612e-01
   5.08825102e-01]
  [3.39903344e-01 9.87543709e-01 4.65598644e-01 3.45995707e-01
   9.58713181e-01]]

 [[5.02084001e-01 2.56200969e-01 5.49844259e-01 4.43027181e-01
   6.01245766e-01]
  [7.90837488e-01 2.28636092e-01 2.26966491e-01 8.38302158

In [ ]:
print(event["image"].numpy())

[ 197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  266.32901016  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  261.34530654  197.11325851  197.11325851
  197.11325851  263.76361099 1348.25107589  298.85057845  197.11325851
  197.11325851  197.11325851  286.39696517  197.11325851  197.11325851
  197.